# 🫀 Prédiction de Maladies Cardiaques
## Master 1 IA & Data Science — UMEF University — 2026
**Auteur :** Mouhamadou Moustapha BA  
**Cours :** Généralités sur les IA  
**Dataset :** Heart Disease UCI — johnsmith88 (Kaggle)  
**Objectif :** Prédire la présence d'une maladie cardiaque à partir de données cliniques

## 0. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports & Configuration

Chargement de toutes les bibliothèques nécessaires et définition des variables globales du projet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix,
                             classification_report, roc_curve, auc)

# ─── CONFIGURATION ───────────────────────────────────────────────────
NUM_COLS     = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CAT_COLS     = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
TARGET       = 'target'
RANDOM_STATE = 42
OUTPUT_DIR   = '/content/drive/MyDrive/heart_disease_ml/output'
MODELS_DIR   = '/content/drive/MyDrive/heart_disease_ml/models'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print('✅ Configuration chargée !')

## 2. Chargement des données

Le dataset **Heart Disease UCI** (johnsmith88) contient 1 025 lignes initiales.  
Après suppression des doublons issus de la fusion de 4 hôpitaux, il reste **302 patients uniques**.

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/heart_disease_ml/heart.csv') \
       .drop_duplicates() \
       .reset_index(drop=True)

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 3. Exploration initiale

Vérification des dimensions, valeurs manquantes et statistiques descriptives.

In [ ]:
print('=== SHAPE ===')
print(df.shape)

print('\n=== VALEURS MANQUANTES ===')
print(df.isnull().sum())

print('\n=== STATISTIQUES DESCRIPTIVES ===')
df.describe()

## 4. Analyse Exploratoire (EDA)

Visualisation de la distribution de la variable cible, de l'âge par classe et de la répartition par sexe.

> **Note :** Dans ce dataset, `target=1` = Sain et `target=0` = Malade.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution cible
counts = df[TARGET].value_counts()
axes[0].bar(['Sain (1)', 'Malade (0)'],
            [counts.get(1, 0), counts.get(0, 0)],
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Sains vs Malades')
for i, v in enumerate([counts.get(1, 0), counts.get(0, 0)]):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=12)

# Age par classe
for val, color, label in [(0, '#e74c3c', 'Malade'), (1, '#2ecc71', 'Sain')]:
    axes[1].hist(df[df[TARGET]==val]['age'], bins=20,
                 alpha=0.6, color=color, label=label)
axes[1].set_title("Âge par classe")
axes[1].legend()

# Maladie par sexe
df.groupby(['sex', TARGET]).size().unstack().plot(
    kind='bar', ax=axes[2],
    color=['#e74c3c', '#2ecc71'],
    title='Maladie par sexe', rot=0)
axes[2].set_xticklabels(['Femme', 'Homme'])
axes[2].legend(['Malade', 'Sain'])

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Image sauvegardée')

## 5. Matrice de corrélation

Analyse des corrélations entre les features et la variable cible.  
Les features les plus corrélées avec `target` guideront l'interprétation des modèles.

In [ ]:
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(df.corr(numeric_only=True), dtype=bool))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', mask=mask, linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Image sauvegardée')

print('\n📊 Top features corrélées avec target :')
corr_target = df.corr(numeric_only=True)[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(corr_target.head(8))

## 6. Preprocessing

- **Split train/test :** 80% / 20% avec `stratify=y` pour préserver la distribution
- **StandardScaler** sur les colonnes numériques (échelles très différentes)
- **Passthrough** sur les colonnes catégorielles (déjà encodées en 0/1)

In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', 'passthrough',   CAT_COLS)
])

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Distribution train : {y_train.value_counts().to_dict()}')
print(f'Distribution test  : {y_test.value_counts().to_dict()}')

## 7. Entraînement des modèles

Trois algorithmes de classification sont comparés :
- **Logistic Regression** — modèle linéaire interprétable
- **Random Forest** — ensemble de décision robuste
- **SVM** — séparateur à vaste marge

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'SVM'                : SVC(probability=True, random_state=RANDOM_STATE)
}

results = {}
for name, model in models.items():
    pipe   = Pipeline([('pre', preprocessor), ('clf', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        'accuracy' : accuracy_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'model'    : pipe,
        'y_pred'   : y_pred
    }
    print(f"{name:<25} | Acc: {results[name]['accuracy']:.4f} | "
          f"F1: {results[name]['f1']:.4f} | "
          f"Precision: {results[name]['precision']:.4f} | "
          f"Recall: {results[name]['recall']:.4f}")

## 8. Validation croisée (5 folds)

La validation croisée permet d'évaluer la stabilité des modèles sur différents sous-ensembles de données.  
Un écart-type faible indique un modèle **stable et généralisable**.

In [ ]:
print('=== VALIDATION CROISÉE (5 FOLDS) ===\n')
for name, res in results.items():
    cv = cross_val_score(res['model'], X, y, cv=5, scoring='accuracy')
    print(f"{name:<25} | CV Mean: {cv.mean():.4f} | Std: ±{cv.std():.4f}")

## 9. Optimisation — GridSearchCV

Recherche des meilleurs hyperparamètres pour chaque modèle par validation croisée 5 folds.  
La métrique d'optimisation est l'**accuracy**.

In [ ]:
param_grids = {
    'Logistic Regression': {
        'clf__C'     : [0.01, 0.1, 1, 10],
        'clf__solver': ['lbfgs', 'liblinear']
    },
    'Random Forest': {
        'clf__n_estimators'     : [100, 200, 300],
        'clf__max_depth'        : [None, 5, 10],
        'clf__min_samples_split': [2, 5]
    },
    'SVM': {
        'clf__C'     : [0.1, 1, 10],
        'clf__kernel': ['rbf', 'linear']
    }
}

best_results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])
    grid = GridSearchCV(pipe, param_grids[name],
                        cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)
    best_results[name] = {
        'accuracy' : accuracy_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'model'    : grid.best_estimator_,
        'y_pred'   : y_pred,
        'params'   : grid.best_params_
    }
    print(f"{name:<25} | Acc: {best_results[name]['accuracy']:.4f} | {grid.best_params_}")

best_name   = max(best_results, key=lambda x: best_results[x]['accuracy'])
best_model  = best_results[best_name]['model']
y_pred_best = best_results[best_name]['y_pred']
print(f'\n👑 Meilleur modèle : {best_name}')

## 10. Sauvegarde des modèles

Sauvegarde du meilleur modèle et des 3 modèles optimisés au format `.pkl` sur Google Drive.

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

# Meilleur modèle
joblib.dump(best_model, f'{MODELS_DIR}/best_model.pkl')
print(f'✅ best_model.pkl sauvegardé → {best_name}')

# Les 3 modèles
for name, res in best_results.items():
    filename = name.lower().replace(' ', '_')
    joblib.dump(res['model'], f'{MODELS_DIR}/{filename}.pkl')
    print(f'✅ {filename}.pkl sauvegardé')

## 11. Comparaison des modèles (graphique)

Visualisation comparative des 4 métriques (Accuracy, F1-Score, Precision, Recall) pour les 3 modèles après optimisation GridSearch.

In [ ]:
names = list(best_results.keys())
x     = np.arange(len(names))
w     = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w*1.5, [best_results[n]['accuracy']  for n in names], w, label='Accuracy',  color='#3498db')
ax.bar(x - w*0.5, [best_results[n]['f1']        for n in names], w, label='F1-Score',  color='#e74c3c')
ax.bar(x + w*0.5, [best_results[n]['precision'] for n in names], w, label='Precision', color='#2ecc71')
ax.bar(x + w*1.5, [best_results[n]['recall']    for n in names], w, label='Recall',    color='#f39c12')

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0.6, 1.0)
ax.legend()
ax.set_title('Comparaison des modèles')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Image sauvegardée')

## 12. Matrice de confusion

La matrice de confusion du meilleur modèle révèle :
- **Vrais Positifs (VP)** : patients sains correctement identifiés
- **Vrais Négatifs (VN)** : patients malades correctement identifiés  
- **Faux Négatifs (FN)** : patients malades prédits comme sains → **erreur critique en médecine**

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malade', 'Sain'],
            yticklabels=['Malade', 'Sain'])
plt.title(f'Matrice de Confusion — {best_name}')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, y_pred_best,
      target_names=['Malade', 'Sain']))
print('✅ Image sauvegardée')

## 13. Courbes ROC

La courbe ROC mesure la capacité du modèle à distinguer malades et sains à différents seuils.  
L'**AUC (Area Under the Curve)** résume la performance — plus elle est proche de 1, meilleur est le modèle.

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71']

for (name, res), color in zip(best_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['model'].predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, color=color,
             label=f'{name} (AUC={auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Baseline')
plt.title('Courbes ROC')
plt.xlabel('Taux Faux Positifs (FPR)')
plt.ylabel('Taux Vrais Positifs (TPR)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Image sauvegardée')

## 14. Prévision sur des patients

### Partie 1 — Patients réels du dataset
Prédiction sur 5 patients réels extraits du jeu de test.

### Partie 2 — Patients fictifs sénégalais
Simulation sur 5 profils cliniques représentatifs d'une population africaine.

> **Rappel :** `risque = (1 - prob_sain) × 100` car `target=1=Sain` dans ce dataset.

In [ ]:
# ════════════════════════════════════════
# PARTIE 1 — PATIENTS RÉELS DU DATASET
# ════════════════════════════════════════
print('📊 PARTIE 1 : PATIENTS RÉELS')
print('=' * 55)

for i in range(5):
    patient = X_test.iloc[[i]]
    prob    = best_model.predict_proba(patient)[0][1]
    pred    = best_model.predict(patient)[0]
    reel    = y_test.iloc[i]
    risque  = (1 - prob) * 100  # target=1=Sain
    predit  = '✅ Sain'    if pred == 1 else '🔴 Malade'
    reel_lb = '✅ Sain'    if reel == 1 else '🔴 Malade'
    correct = '✅ Correct' if pred == reel else '❌ Incorrect'

    print(f'\n👤 Patient #{i+1}')
    print(f'  Age        : {int(patient["age"].values[0])} ans')
    print(f'  Sexe       : {"Homme" if int(patient["sex"].values[0])==1 else "Femme"}')
    print(f'  Cholestérol: {int(patient["chol"].values[0])} mg/dl')
    print(f'  Risque     : {risque:.1f}%')
    print(f'  Prédit     : {predit}')
    print(f'  Réel       : {reel_lb}')
    print(f'  Verdict    : {correct}')

# ════════════════════════════════════════
# PARTIE 2 — PATIENTS FICTIFS SÉNÉGALAIS
# ════════════════════════════════════════
print('\n📊 PARTIE 2 : PATIENTS FICTIFS SÉNÉGALAIS')
print('=' * 55)

profils = {
    'Mamadou Diallo — Homme 63 ans — Profil à risque': {
        'age':63,'sex':1,'cp':0,'trestbps':160,'chol':340,
        'fbs':1,'restecg':2,'thalach':110,'exang':1,
        'oldpeak':3.5,'slope':0,'ca':3,'thal':3
    },
    'Aissatou Ndiaye — Femme 45 ans — Profil sain': {
        'age':45,'sex':0,'cp':2,'trestbps':110,'chol':180,
        'fbs':0,'restecg':0,'thalach':175,'exang':0,
        'oldpeak':0.2,'slope':2,'ca':0,'thal':2
    },
    'Fatou Sow — Femme 65 ans — Profil intermédiaire': {
        'age':65,'sex':0,'cp':1,'trestbps':145,'chol':260,
        'fbs':1,'restecg':1,'thalach':130,'exang':0,
        'oldpeak':1.8,'slope':1,'ca':1,'thal':2
    },
    'Ibrahima Mbaye — Homme 40 ans — Profil jeune sain': {
        'age':40,'sex':1,'cp':2,'trestbps':115,'chol':195,
        'fbs':0,'restecg':0,'thalach':180,'exang':0,
        'oldpeak':0.1,'slope':2,'ca':0,'thal':2
    },
    'Rokhaya Diop — Femme 35 ans — Profil très sain': {
        'age':35,'sex':0,'cp':3,'trestbps':105,'chol':170,
        'fbs':0,'restecg':0,'thalach':190,'exang':0,
        'oldpeak':0.0,'slope':2,'ca':0,'thal':2
    },
}

for desc, profil in profils.items():
    patient = pd.DataFrame([profil])
    prob    = best_model.predict_proba(patient)[0][1]
    pred    = best_model.predict(patient)[0]
    risque  = (1 - prob) * 100
    statut  = '✅ Sain'  if pred == 1 else '🔴 Malade'
    emoji   = '🔴' if risque >= 70 else ('🟡' if risque >= 40 else '✅')
    print(f'\n👤 {desc}')
    print(f'  Risque cardiaque : {risque:.1f}% {emoji} → {statut}')